# Export organized spike trains and event logs

Run this **once on the computer with the original `Full_Dataset/` directory**. This notebook loads the raw files exactly as the original organizer did, and saves **only** `spikes`, `labels`, and `event_logs`. It does not calculate trial-aligned rates, residuals, condition averages, RRR or RDMs. The data are compressed into `Figure5_organized_inputs.npz`.

The importer is `Figure5_OrganizedInputs_Analysis.ipynb`. If your original notebook is already running and `spikes`, `event_logs`, `labels` are defined, you only need to run the final export cell below in that kernel.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import scipy.io
import mat73

DATA_ROOT = Path('./Full_Dataset')
if not DATA_ROOT.exists():
    raise FileNotFoundError('Original Full_Dataset/ is required only for the one-time export')
PARTICIPANTS = ['YFF','YFI','YFJ','YFK','YFL','YFM','YFR','YFS','YFT','YFU']
REGIONS = ['acc','amy','ent','hpc','ofc','para-hpc','thal']
NUM_GROUPS = [[n] for n in range(1,11)]


In [2]:
def load_mat(filepath):
    import mat73
    import scipy.io as sio
    try:
        return mat73.loadmat(filepath), 'mat73'
    except Exception:
        pass
    try:
        return sio.loadmat(filepath, squeeze_me=True, struct_as_record=False), 'scipy'
    except Exception as e:
        raise RuntimeError(f"Cannot load {filepath}: {e}")

def extract_behav_array(data, loader, key='behavs_array'):
    raw = data[key]
    if loader == 'mat73':
        if isinstance(raw, dict):
            return np.column_stack([np.array(c, dtype=float).ravel() for c in raw.values()])
        return np.array(raw, dtype=float)
    elif loader == 'scipy':
        if hasattr(raw, 'dtype') and raw.dtype.names:
            return np.column_stack([raw[n].ravel().astype(float) for n in raw.dtype.names])
        arr = np.array(raw, dtype=float)
        return arr.reshape(-1, 1) if arr.ndim == 1 else arr

def extract_sparse_spikes(data, loader):
    from scipy.sparse import csr_matrix, issparse
    raw = None
    for key in ['spikesDot', 'spikes']:
        if key in data:
            raw = data[key]; break
    if raw is None:
        raise KeyError("No spikes key found.")
    if loader == 'mat73':
        if isinstance(raw, np.ndarray):
            return csr_matrix(raw)
        if issparse(raw): return raw.tocsr()
    elif loader == 'scipy':
        if issparse(raw): return raw.tocsr()
        return csr_matrix(np.array(raw))
    raise ValueError("Cannot parse spikes.")

def extract_feature(data):
    presen_start = data.tDot
    presen_end = data.presentationEnd
    resp_start = data.tPress1
    resp_end = data.choiceEnd
    dot_num = data.dotNum
    correct = data.correct
    error = data.dotNum - data.givenAnsw

    results = {
        'presen_start': np.round(presen_start).astype(np.int64),
        'presen_end': np.round(presen_end).astype(np.int64),
        'resp_start': np.round(resp_start).astype(np.int64),
        'resp_end': np.round(resp_end).astype(np.int64),
        'dot_num': dot_num,
        'correct': correct,
        'error': error
    }
    return results

In [3]:
event_logs = []
spikes = []
labels = []
for sub in PARTICIPANTS:
    if sub == 'YFK':
        for ses in ['session02']:
            event_log = pd.read_csv(f'{DATA_ROOT}/event_logs/{sub}_event_log_{ses}.csv')
            try:
                spike_data = load_mat(f'{DATA_ROOT}/{sub}/{ses}/dot/spikesDot.mat')
            except:
                spike_data = load_mat(f'{DATA_ROOT}/{sub}/{ses}/dot/spikesDot.mat')
            spike_array = extract_sparse_spikes(spike_data[0], spike_data[1]).toarray()
            spike_label = np.array(spike_data[0]['regionsVect'])[:,0]
            spike_label[spike_label == 'OFC/vmPFC'] = 'ofc'
            spike_label[spike_label == 'thalamus'] = 'thal'

            event_logs.append(extract_feature(event_log))
            spikes.append(spike_array)
            labels.append(spike_label)
    elif sub == 'YFL':
        event_log = pd.read_csv(f'{DATA_ROOT}/event_logs/{sub}_event_log.csv').iloc[:119]
        try:
            spike_data = load_mat(f'{DATA_ROOT}/{sub}/dot/spikes.mat')
        except:
            spike_data = load_mat(f'{DATA_ROOT}/{sub}/dot/spikesDot.mat')
        spike_array = extract_sparse_spikes(spike_data[0], spike_data[1]).toarray()
        spike_label = np.array(spike_data[0]['regionsVect'])[:,0]
        spike_label[spike_label == 'OFC/vmPFC'] = 'ofc'
        spike_label[spike_label == 'thalamus'] = 'thal'

        event_logs.append(extract_feature(event_log))
        spikes.append(spike_array)
        labels.append(spike_label)
    else:
        event_log = pd.read_csv(f'{DATA_ROOT}/event_logs/{sub}_event_log.csv')
        try:
            spike_data = load_mat(f'{DATA_ROOT}/{sub}/dot/spikes.mat')
        except:
            spike_data = load_mat(f'{DATA_ROOT}/{sub}/dot/spikesDot.mat')
        spike_array = extract_sparse_spikes(spike_data[0], spike_data[1]).toarray()
        spike_label = np.array(spike_data[0]['regionsVect'])[:,0]
        spike_label[spike_label == 'OFC/vmPFC'] = 'ofc'
        spike_label[spike_label == 'thalamus'] = 'thal'

        event_logs.append(extract_feature(event_log))
        spikes.append(spike_array)
        labels.append(spike_label)
labels_concat = np.concatenate(labels)

trial_idx = [[np.concatenate([np.where(event_logs[s]['dot_num'] == num)[0] for num in NUM_GROUPS[n]]) for n in range(len(NUM_GROUPS))] for s in range(len(event_logs))]
# trial_idx = [[np.where(event_logs[s]['dot_num'] == num)[0] for num in range(1, 11)] for s in range(len(event_logs))]
region_idx = [np.where(labels_concat == r)[0] for r in REGIONS]
region_indices = [[np.where(labels[s] == r)[0] for r in REGIONS] for s in range(len(event_logs))]
neuron_idx = [spikes[s].shape[0] for s in range(len(event_logs))]
neuron_idx_cumsum = np.cumsum(neuron_idx)
neuron_idx_cumsum = np.concatenate([[0], neuron_idx_cumsum])

In [ ]:
# Store ONLY the organized spike matrices, region labels and per-trial event arrays.
# Archive is read with allow_pickle=False (no pickle/object arrays required).
EVENT_KEYS = ('presen_start','presen_end','resp_start','resp_end','dot_num','correct','error')
payload = {'metadata_json': np.array(json.dumps({
    'schema': 'figure5-organized-inputs-v1',
    'participants': PARTICIPANTS,
    'regions': REGIONS,
    'event_keys': list(EVENT_KEYS),
    'description': 'Raw-time-base spike matrices and trial/event arrays; no RRR preprocessing'
}))}
for i, name in enumerate(PARTICIPANTS):
    spike_matrix = np.asarray(spikes[i])
    label_array = np.asarray(labels[i]).astype(str)
    assert spike_matrix.ndim == 2 and len(label_array) == spike_matrix.shape[0], name
    payload[f'spikes_{i}'] = spike_matrix
    payload[f'labels_{i}'] = label_array
    for key in EVENT_KEYS:
        val = np.asarray(event_logs[i][key])
        if val.dtype.kind in 'OU':
            raise TypeError(f'{name}/{key} has unsupported object dtype; review before export')
        payload[f'event_{i}_{key}'] = val
    print(name, spike_matrix.shape, len(event_logs[i]['dot_num']))

out = Path('Figure5_organized_inputs.npz')
np.savez_compressed(out, **payload)
print('Saved:', out.resolve(), 'Size MiB:', round(out.stat().st_size/2**20, 1))